# Databricks verification Model training & S3 Integration

This notebook demonstrates how to run the verification training job directly in **Databricks** by downloading the processed dataset (`train_database.h5`) directly from your S3 bucket (Option A).

## Step 1: Install Dependencies
Run this cell to install required libraries (`boto3`, `h5py`, `mne`, `scikit-learn`) on the active cluster node.

In [ ]:
!pip install boto3 h5py mne scikit-learn tensorflow

## Step 2: Configure S3 & Download Dataset
Retrieve the 170MB balanced dataset (`train_database.h5`) from S3 using credentials. 

> **Tip:** For production, it's recommended to mount S3 directly using Databricks IAM Instance Profiles or read credentials from Databricks Secrets.

In [ ]:
import os
import boto3

# Set AWS access credentials if required, or leave blank if your cluster uses an attached IAM role
aws_access_key = dbutils.widgets.text("aws_access_key", "")
aws_secret_key = dbutils.widgets.text("aws_secret_key", "")
region = "eu-north-1"

access_key = dbutils.widgets.get("aws_access_key")
secret_key = dbutils.widgets.get("aws_secret_key")

if access_key and secret_key:
    print("[*] Authenticating with provided access keys...")
    s3 = boto3.client(
        's3',
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name=region
    )
else:
    print("[*] Authenticating using attached cluster IAM role...")
    s3 = boto3.client('s3', region_name=region)

bucket_name = 'seizure-detection--s-roy'
s3_key = 'data/train_database.h5'
local_h5_path = '/tmp/train_database.h5'

print(f"[*] Downloading train_database.h5 from s3://{bucket_name}/{s3_key}...")
s3.download_file(bucket_name, s3_key, local_h5_path)
print(f"✅ Dataset successfully downloaded locally to: {local_h5_path}")

## Step 3: Load Dataset & Stratified Splits
Read the HDF5 data windows and split into train/validation chunks.

In [ ]:
import h5py
import numpy as np
from sklearn.model_selection import train_test_split

with h5py.File(local_h5_path, 'r') as h5f:
    X = np.array(h5f['X'], dtype=np.float32)
    y = np.array(h5f['y'], dtype=np.int32)
    
print(f"Dataset raw signals shape: {X.shape}")
print(f"Dataset labels shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

## Step 4: Compile MobileNetV2 2D-CNN & Run verification
Build the 2D-CNN model architecture, compile, and run training.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = (10, 256) # 10 channels, 256 samples (2.0s @ 128Hz)
inputs = layers.Input(shape=input_shape)

# 1D to 2D expansion bridge
x = layers.Reshape((10, 256, 1))(inputs)
x = layers.Conv2D(3, (1, 1), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Resizing(224, 224)(x)

# MobileNetV2 Frozen backbone
base = tf.keras.applications.MobileNetV2(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3)
)
base.trainable = False
x = base(x, training=False)

# Output layers
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(2, activation="linear")(x)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.summary()

print("[*] Launching 1-epoch verification run...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=1,
    batch_size=32
)
print("✅ Verification model successfully trained inside Databricks!")